In [2]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel
from trl import RewardTrainer, RewardConfig
from datasets import load_dataset

# 第一步：合并 SFT 的 LoRA 补丁
需要将之前的“补丁”和“底座”融为一体，生成一个新的、完整的模型文件夹。

In [5]:
base_model_path = "./model/" # 原始底座，如 Qwen, Llama
sft_lora_path = "./lora_results/" # 你之前的 SFT 结果

# 1. 加载底座
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path, 
    torch_dtype=torch.float16, 
    device_map="auto"
)

# 2. 挂载 SFT LoRA
model = PeftModel.from_pretrained(base_model, sft_lora_path)

# 3. 合并并卸载 (核心步骤)
# 这会将 LoRA 的权重永久性地加到底座权重中
merged_model = model.merge_and_unload()

# 4. 保存为“RM 的专用底座”
merged_model.save_pretrained("./model_merged_for_rm")
tokenizer = AutoTokenizer.from_pretrained(base_model_path)
tokenizer.save_pretrained("./model_merged_for_rm")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./model_merged_for_rm/tokenizer_config.json',
 './model_merged_for_rm/chat_template.jinja',
 './model_merged_for_rm/tokenizer.json')

# 第二步：使用 trl 库中的 RewardTrainer来训练奖励模型

In [6]:
# 1. 导入并处理数据集
# 导入数据集
raw_dataset = load_dataset("json", data_files='RLHF_dataset.json')

# 将数据集切分为训练集 (90%) 和验证集 (10%)，用于监控模型是否真正学会了打分
split_dataset = raw_dataset["train"].train_test_split(test_size=0.1, seed=42)

# 对数据做字符串的拼接！
def format_dataset(examples):
    formatted = {
        "chosen": [],
        "rejected": []
    }
    for p, c, r in zip(examples["prompt"], examples["chosen"], examples["rejected"]):
        formatted["chosen"].append(f"User: {p}\nAssistant: {c}")
        formatted["rejected"].append(f"User: {p}\nAssistant: {r}")
    return formatted

# 使用 map 覆盖原有的 chosen 和 rejected 列
formatted_dataset = split_dataset.map(
    format_dataset,
    batched=True   # 一次性处理一批数据
)

formatted_dataset['train'][0]

{'prompt': '写一段 Python 冒泡排序。',
 'chosen': 'User: 写一段 Python 冒泡排序。\nAssistant: 好的，这是一种基础的排序算法，代码如下：...',
 'rejected': 'User: 写一段 Python 冒泡排序。\nAssistant: 去百度搜一下就知道了，干嘛问我。'}

In [7]:
# 2. 模型加载，引入 LoRA

model_path = "./model_merged_for_rm"
tokenizer = AutoTokenizer.from_pretrained(model_path)
# 注意：微调时通常需要 pad_token。如果模型没有，我们指定 eos_token 为 pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 加载模型并替换分类头 (从生成者变为裁判)
print(">>> 正在加载底座模型并初始化分类头...")
# num_labels=1: 丢弃原有的 lm_head，换上一个输出维度为 1 的 score 线性层
model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=1,
    torch_dtype=torch.bfloat16, # 推荐使用 bfloat16 节省显存
    device_map="auto"           # 自动分配显存
)

# 确保模型的 pad_token_id 与 tokenizer 一致
model.config.pad_token_id = tokenizer.pad_token_id

# 挂载第二层 LoRA 补丁 (专门用于学习打分)
print(">>> 正在配置 RM 专属 LoRA...")
rm_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"], # 覆盖主要注意力层
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS", # 关键：告诉 PEFT 这是一个序列分类任务
)
model = get_peft_model(model, rm_lora_config)
model.print_trainable_parameters() # 打印可训练参数比例


`torch_dtype` is deprecated! Use `dtype` instead!


>>> 正在加载底座模型并初始化分类头...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: ./model_merged_for_rm
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


>>> 正在配置 RM 专属 LoRA...
trainable params: 2,180,608 || all params: 1,545,896,448 || trainable%: 0.1411


In [8]:
# 3. 配置并启动 RewardTrainer
training_args = RewardConfig(
    output_dir="./RLHF_RM_Lora",
    per_device_train_batch_size=2,
    learning_rate=1e-5,
    max_length=512, # 截断长度在这里设置！
)

trainer = RewardTrainer(
    model=model,
    processing_class=tokenizer, 
    args=training_args,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["test"], 
)
# 开始训练！
trainer.train()

# 7. 保存最终的 RM LoRA 补丁
print(f">>> 训练完成，正在保存 Reward Model LoRA 到 RLHF_RM")
trainer.save_model("./RLHF_RM_Lora")
tokenizer.save_pretrained("./RLHF_RM_Lora")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 151646}.
/opt/miniconda3/envs/torch/lib/python3.10/site-packages/torch/utils/data/dataloader.py:774: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss


>>> 训练完成，正在保存 Reward Model LoRA 到 RLHF_RM


('./RLHF_RM_Lora/tokenizer_config.json',
 './RLHF_RM_Lora/chat_template.jinja',
 './RLHF_RM_Lora/tokenizer.json')

In [9]:
# 4. 合并补丁，生成 RM 模型
# 1. 定义路径
rm_base_path = "./model_merged_for_rm" # 训练 RM 时用的底座
rm_lora_path = "./RLHF_RM_Lora"           # 刚刚训练出来的 RM LoRA 补丁

# 2. 加载 RM 的底座 (注意是分类模型类，且带上 num_labels=1)
print(">>> 正在加载 RM 底座...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    rm_base_path,
    num_labels=1,
    torch_dtype=torch.bfloat16,
    device_map="cpu" # 合并操作建议在 CPU 上进行，防止显存 OOM
)

# 3. 挂载 RM 的 LoRA 补丁
print(">>> 正在挂载 RM LoRA...")
model = PeftModel.from_pretrained(base_model, rm_lora_path)

# 4. 执行物理合并
print(">>> 正在合并权重 (这可能需要几分钟)...")
merged_rm_model = model.merge_and_unload()

# 5. 保存最终的独立裁判模型
final_rm_path = "./final_reward_model_merged"
print(f">>> 保存最终 RM 模型到 {final_rm_path}")
merged_rm_model.save_pretrained(final_rm_path)

# 别忘了保存 tokenizer
tokenizer = AutoTokenizer.from_pretrained(rm_base_path)
tokenizer.save_pretrained(final_rm_path)
print("✅ RM 合并完成！准备就绪！")

>>> 正在加载 RM 底座...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: ./model_merged_for_rm
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


>>> 正在挂载 RM LoRA...
>>> 正在合并权重 (这可能需要几分钟)...
>>> 保存最终 RM 模型到 ./final_reward_model_merged


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ RM 合并完成！准备就绪！


# 第三步：强化学习
核心目的：利用上一步的“裁判（RM）”，通过强化学习不断调整“选手（SFT 模型）”的权重，让它生成的回答得分越来越高。


In [ ]:
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer
from transformers.trainer_utils import set_seed
from datasets import Dataset

# 基础配置和路径

In [32]:
# 设置随机种子保证实验可复现
set_seed(42)

# 1. 配置路径与超参数
SFT_MODEL_PATH = "./model_merged_for_rm"       # 之前合并好的 SFT 模型（作为 Actor/Ref 的基座）
RM_MODEL_PATH = "./final_reward_model_merged" # 刚才合并出来的独立裁判模型
PROMPT_DATASET = "RLHF_dataset.json"         # PPO 阶段不需要 chosen/rejected，只需要 prompt 即可
   
# 2. 加载 Tokenizer (显式处理特殊 Token，消除警告)
print(">>> 正在加载 Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_PATH)

# 显式设置 PAD Token (如果没有的话)
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token
#     # 强制对齐，避免后续警告
#     tokenizer.pad_token_id = tokenizer.eos_token_id

>>> 正在加载 Tokenizer...


# 加载奖励模型

In [33]:
# 3. 加载 Reward Model
print(">>> 正在加载 Reward Model...")
rm_model = AutoModelForSequenceClassification.from_pretrained(
    RM_MODEL_PATH, 
    num_labels=1, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)
rm_model.eval()

tokenizer.pad_token = tokenizer.eos_token
rm_model.config.pad_token_id = tokenizer.pad_token_id
# 4. 定义 Reward 函数
def reward_func(completions, **kwargs):
    prompts = kwargs["prompts"]
    texts = [p + c for p, c in zip(prompts, completions)]
    inputs = tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        return_tensors="pt"
    ).to(rm_model.device)
    
    with torch.no_grad():
        scores = rm_model(**inputs).logits.squeeze(-1)
    
    return scores.tolist()

>>> 正在加载 Reward Model...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

# 准备数据集

In [34]:
# 5. 准备数据 (增加调试打印，确保数据加载正确)
print(">>> 正在准备数据集...")
import json
with open(PROMPT_DATASET, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# 构建 prompts 列表
prompts_list = []
for item in raw_data:
    # 容错处理 key 名
    prompt_key = next((k for k in item.keys() if 'prompt' in k), 'prompt')
    prompt_text = item[prompt_key]
    formatted_prompt = f"User: {prompt_text}\nAssistant: "
    prompts_list.append(formatted_prompt)

# 构建 Dataset (这一步确保 __len__ 是存在的)
dataset = Dataset.from_dict({"prompt": prompts_list})

# 【关键调试】打印数据集大小，确认加载成功
print(f">>> 数据集加载完成，共 {len(dataset)} 条数据")

>>> 正在准备数据集...
>>> 数据集加载完成，共 3 条数据


# 配置LoRA

In [35]:
# 6. 配置 LoRA
lora_config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules=["q_proj", "v_proj"], 
    bias="none", 
    task_type="CAUSAL_LM"
)

# 配置 GRPO

In [36]:
# 7. 配置 GRPO (核心修复：显式设置 max_steps 或 确保 num_train_epochs 生效)
# 计算一下大概的 max_steps (可选，也可以直接硬编码一个数字)
batch_size = 1
accumulation_steps = 4  # 梯度存多少步再更新
num_epochs = 1
# 公式：steps = (数据量 / batch_size) * epochs / accumulation_steps
estimated_max_steps = int((len(dataset) / batch_size) * num_epochs / accumulation_steps)
# 防止计算出 0，至少给 10 步
estimated_max_steps = max(estimated_max_steps, 10)

training_args = GRPOConfig(
    output_dir="./final_rlhf_llm",
    learning_rate=1.41e-5,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=accumulation_steps,
    max_completion_length=8, 
    
    # 【核心修复 1】要么用这个：显式指定训练轮数
    # num_train_epochs=num_epochs,
    
    max_steps=estimated_max_steps,  
    
    # 【关键加速 2】减小组内生成数量 (这是最重要的！)
    num_generations=2,         # 默认是 8，改成 2 或 4，速度直接翻倍/翻4倍
    
    # 【核心修复 2】要么用这个：直接指定最大步数 (二选一，推荐用上面的 num_train_epochs)
    # max_steps=estimated_max_steps, 
    
    logging_steps=1, # 改小一点，方便看进度
    bf16=True,
    remove_unused_columns=False, # 有时候需要这个防止数据被意外删除
)


# 初始化 Trainer，开始训练

In [37]:
# 8. 初始化 Trainer
print(">>> 正在初始化 Trainer...")
trainer = GRPOTrainer(
    model=SFT_MODEL_PATH,
    reward_funcs=reward_func,
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,
)

# 9. 开练！
print(">>> 开始训练！")
trainer.train()

print(">>> 训练完成，保存模型...")
trainer.save_model("./final_rlhf_llm")

>>> 正在初始化 Trainer...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 151646, 'pad_token_id': 151643}.


>>> 开始训练！


/opt/miniconda3/envs/torch/lib/python3.10/site-packages/torch/utils/data/dataloader.py:774: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.000000
2,0.000000
3,0.000000
4,-0.000000
5,0.000000
6,0.000000
7,0.000000
8,0.000000
9,0.000000
10,0.000000


>>> 训练完成，保存模型...
